In [2]:
import fsspec
import xarray as xr
import scipy.spatial
import numpy as np
import pandas as pd 
import os
import argparse
from datetime import date
import datetime
from calculations.calculations import vapor_pressure
from calculations.calculations import wind_tot
from calculations.calculations import rel_hum
import regionmask
import geopandas as gpd
import scipy.stats as stats
from ERA5_functions import *

In [2]:
ds = xr.open_mfdataset('/data/keeling/a/rytam2/a/iema_output/variables_202510210102.nc')
precip = ds.precip.sel(time=ds.time.dt.year <= 2024)

fpath = '/data/keeling/a/rytam2/a/iema_output/arcgis_toprocess/' #for output to arcgis

1mo_spi: total fits = 10years *12months*29*27 (all loc)
12mo_spi: total fits = 365

gamma fit 31days*10years for each month 
gamma fit 365*10 

#### Tier 4 - SPI
raw: 
- 1-month SPI - individual day deviation from a normal Jan: gamma fit is conducted for every day with calculated from
- 12-month SPI - individual day deviation from a normal year

365

monthly: 
- total mod/ext dry days (1mo SPI) each month in each year 
- total mod/ext wet days (1mo SPI) each month in each year
- average precip in each month across all years (same monthly average for each year)
- **total mod/ext dry days (12mo SPI)**
- **total mod/ext wet days (12mo SPI)**
- **average precip in each year across all** 
- ~~AVG 1-month SPI~~ seems to not make sense? 
- ~~AVG 12-month SPI~~ seems to not make sense?

yearly 

summary: 
- total mod/ext dry day (sum from monthly)
- 

*** Will revisit on how to calculate
- gamma distribution??
- check how the distrubition works and how to check dataset distribution

ppt_mo_mean = ds.groupby('time.month').mean('time')
ppt_anom = ds.groupby('time.month') - ppt_mo_mean 
sigma_p = ds.groupby('time.month').std(dim='time') 
spi = ((ppt.groupby('time.month') - ppt_mo_mean).groupby('time.month'))/sigma_p

### All fit SPI 

In [6]:
# pre = recent_an['total_precipitation'].rename({'latitude':'lat','longitude':'lon'})
# first .where (precip>0) to remove any negative precip
preci = precip.where(precip > 0, other=np.nan).resample(time='1D').sum()#.isel(lat=1,lon=0).values.flatten() .where(precip > 1e-3, other=np.nan)
preci += 1e-10

In [9]:
# %time data_flat = preci.where(preci > 1e-3, other=np.nan).isel(lat=la, lon=lo).values.flatten()
# data_flat = data_flat[~np.isnan(data_flat)]
# np.unique(data_flat)

In [8]:
%%time 
### daily SPI Gamma Fit
shape_arr = []# np.zeros((29, 27), dtype=float)
scale_arr = []# np.zeros((29, 27), dtype=float)

for la in np.arange(0,29):
    for lo in np.arange(0,27):
        print(la,lo)

        %time data_flat = preci.where(preci > 1e-3, other=np.nan).isel(lat=la, lon=lo).values.flatten()
        data_flat = data_flat[~np.isnan(data_flat)]
        print(len(data_flat))

        # Fit gamma distribution fixing location to 0
        shape, _, scale = stats.gamma.fit(data_flat, floc=0)
        shape_arr.append(shape)
        scale_arr.append(scale)

0 0
CPU times: user 928 ms, sys: 278 ms, total: 1.21 s
Wall time: 1.08 s
1021
0 1
CPU times: user 458 ms, sys: 203 ms, total: 661 ms
Wall time: 442 ms
1014
0 2
CPU times: user 472 ms, sys: 197 ms, total: 668 ms
Wall time: 449 ms
1022
0 3
CPU times: user 432 ms, sys: 217 ms, total: 649 ms
Wall time: 443 ms
1044
0 4
CPU times: user 479 ms, sys: 184 ms, total: 663 ms
Wall time: 444 ms
1048
0 5
CPU times: user 437 ms, sys: 221 ms, total: 658 ms
Wall time: 440 ms
1050
0 6
CPU times: user 473 ms, sys: 186 ms, total: 659 ms
Wall time: 439 ms
1051
0 7
CPU times: user 466 ms, sys: 190 ms, total: 656 ms
Wall time: 437 ms
1059
0 8
CPU times: user 455 ms, sys: 202 ms, total: 658 ms
Wall time: 437 ms
1052
0 9
CPU times: user 481 ms, sys: 176 ms, total: 656 ms
Wall time: 436 ms
1044
0 10
CPU times: user 438 ms, sys: 219 ms, total: 657 ms
Wall time: 437 ms
1044
0 11
CPU times: user 445 ms, sys: 212 ms, total: 657 ms
Wall time: 437 ms
1047
0 12
CPU times: user 445 ms, sys: 212 ms, total: 657 ms
Wall t

In [10]:
gamma_shape = np.tile(np.array(shape_arr).reshape(29,27), (len(preci.time), 1, 1))
gamma_scale = np.tile(np.array(scale_arr).reshape(29,27), (len(preci.time), 1, 1))

In [61]:
stats.norm.ppf(stats.gamma.cdf(1e-8, a=gamma_shape, loc=0, scale=gamma_scale))

array([[[-5.13175624, -5.16333117, -5.16145268, ..., -5.42243332,
         -5.43499513, -5.37952144],
        [-5.15614396, -5.10285837, -5.09732575, ..., -5.39980384,
         -5.32490022, -5.35429575],
        [-5.16755585, -5.11226041, -5.04680613, ..., -5.33493594,
         -5.30976462, -5.27134477],
        ...,
        [-5.01089935, -5.01584868, -4.94203658, ..., -5.1599371 ,
         -5.2219135 , -5.24536791],
        [-5.0562865 , -5.04733141, -5.02626331, ..., -5.16417511,
         -5.20674453, -5.16824305],
        [-5.03607454, -5.04013519, -4.99366565, ..., -5.14370486,
         -5.18134473, -5.21896359]],

       [[-5.13175624, -5.16333117, -5.16145268, ..., -5.42243332,
         -5.43499513, -5.37952144],
        [-5.15614396, -5.10285837, -5.09732575, ..., -5.39980384,
         -5.32490022, -5.35429575],
        [-5.16755585, -5.11226041, -5.04680613, ..., -5.33493594,
         -5.30976462, -5.27134477],
        ...,
        [-5.01089935, -5.01584868, -4.94203658, ..., -

In [11]:
# Step 2: Calculate CDF for each day in original data
probabilities = stats.gamma.cdf(preci, a=gamma_shape, loc=0, scale=gamma_scale)
# Step 3: Transform probabilities to a standard normal distribution (Z-score)
spi_values = stats.norm.ppf(probabilities)

In [14]:
spi_xarray = xr.DataArray(
    spi_values,
    coords=preci.coords,
    dims=preci.dims,
    name='SPI'
)

In [15]:
spi_xarray

<xarray.DataArray 'SPI' (time: 3288, lat: 29, lon: 27)> Size: 21MB
array([[[-2.93816203, -2.95561306, -2.96235783, ..., -1.58623314,
         -1.32755421, -1.22033528],
        [-3.04022572, -2.96774071, -2.94935646, ..., -1.60188799,
         -1.31924672, -1.21169353],
        [-3.08627333, -2.99469027, -2.94520185, ..., -1.65119376,
         -1.37851224, -1.12608059],
        ...,
        [-5.90806618, -5.91424305, -5.83044334, ..., -6.07969807,
         -6.15040096, -6.17733007],
        [-5.96282024, -5.95189316, -5.92722682, ..., -6.08448594,
         -6.13382063, -6.08951798],
        [-5.94239881, -5.94581863, -5.89053102, ..., -6.06211703,
         -6.10507126, -6.14779285]],

       [[-6.05979383, -6.09535698, -6.09251311, ..., -2.40639559,
         -2.24981509, -2.08961257],
        [-6.08667958, -6.02456747, -6.0170815 , ..., -2.42802216,
         -2.22337436, -2.10519222],
        [-6.09931903, -6.03495442, -5.95759309, ..., -2.49291907,
         -2.2997885 , -2.12629648],
...
        [-3.32598493, -3.30382454, -3.20423784, ..., -1.00322961,
         -0.97618703, -0.95131645],
        [-3.20753121, -3.60876004, -3.49564811, ..., -1.28807735,
         -1.15578997, -1.09726923],
        [-2.93762636, -3.10258749, -3.03915487, ..., -1.47681109,
         -1.31128375, -0.95207791]],

       [[-1.11213767, -1.13699247, -1.25665864, ..., -1.35621506,
         -1.42953444, -1.4167411 ],
        [-0.98623089, -1.13964059, -1.27974119, ..., -1.0324307 ,
         -1.03130553, -1.05570056],
        [-0.99708834, -1.12944891, -1.24471766, ..., -0.65708576,
         -0.70464935, -0.73556999],
        ...,
        [-2.47702324, -2.71620081, -2.7468564 , ..., -1.40829221,
         -1.19436586, -1.01276643],
        [-3.10128252, -3.14085613, -3.33182758, ..., -1.58499368,
         -1.34399758, -1.10939919],
        [-3.51298761, -3.71845929, -3.60439832, ..., -1.69399996,
         -1.41872354, -1.10405283]]], shape=(3288, 29, 27))
Coordinates:
  * lat      (lat) float32 116B 43.25 43.0 42.75 42.5 ... 37.0 36.75 36.5 36.25
  * lon      (lon) float32 108B 267.2 267.5 267.8 268.0 ... 273.2 273.5 273.8
    county   (lat, lon) <U11 34kB dask.array<chunksize=(29, 27), meta=np.ndarray>
  * time     (time) datetime64[ns] 26kB 2016-01-01 2016-01-02 ... 2024-12-31

In [27]:
tdays_moddrought = (spi_xarray < -0.8).sum().rename('TOT_DAYS_MODDROUGHT')#.groupby(['time.year']).sum().stack(time=('year')).rename('TOT_DAYS_MODDROUGHT')
tdays_extdrought = (spi_xarray < -1.6).sum().rename('TOT_DAYS_EXTDROUGHT')#.groupby(['time.year']).sum().stack(time=('year', 'month')).rename('TOT_DAYS_EXTDROUGHT')
tdays_modplv = (spi_xarray > 0.8).sum().rename('TOT_DAYS_MODPLV')#.groupby(['time.year', 'time.month']).sum().stack(time=('year', 'month')).rename('TOT_DAYS_MODPLV')
tdays_extplv = (spi_xarray > 1.6).sum().rename('TOT_DAYS_EXTPLV')#.groupby(['time.year', 'time.month']).sum().stack(time=('year', 'month')).rename('TOT_DAYS_EXTPLV')

# avgdays_moddrought = (spi < -0.8).groupby(['time.year', 'time.month']).mean().stack(time=('year', 'month')).rename('AVG_DAYS_MODDROUGHT')
# avgdays_extdrought = (spi < -1.6).groupby(['time.year', 'time.month']).mean().stack(time=('year', 'month')).rename('AVG_DAYS_EXTDROUGHT')
# avgdays_modplv = (spi > 0.8).groupby(['time.year', 'time.month']).mean().stack(time=('year', 'month')).rename('AVG_DAYS_MODPLV')
# avgdays_extplv = (spi > 1.6).groupby(['time.year', 'time.month']).mean().stack(time=('year', 'month')).rename('AVG_DAYS_EXTPLV')

monthlystat = xr.merge([tdays_moddrought,tdays_extdrought,tdays_modplv,tdays_extplv])
# times = pd.to_datetime(['{}-{:02d}-01'.format(y, m) for y, m in monthlystat['time'].values])
# monthlystat = monthlystat.drop_vars(['time', 'year', 'month']).assign_coords(time=times)

# monthlystat.to_netcdf('/data/keeling/a/rytam2/iema/data/output/monthlystats_spi_latlon_2016-2025.csv')

In [39]:
# sumamry 

tdays_moddrought = (spi_xarray < -0.8).sum(dim='time').rename('TOT_DAYS_MODDROUGHT')
tdays_extdrought = (spi_xarray < -1.6).sum(dim='time').rename('TOT_DAYS_EXTDROUGHT')
tdays_modplv = (spi_xarray > 0.8).sum(dim='time').rename('TOT_DAYS_MODPLV')
tdays_extplv = (spi_xarray > 1.6).sum(dim='time').rename('TOT_DAYS_EXTPLV')


In [49]:
spi_xarray

<xarray.DataArray 'SPI' (time: 3288, lat: 29, lon: 27)> Size: 21MB
array([[[-2.93816203, -2.95561306, -2.96235783, ..., -1.58623314,
         -1.32755421, -1.22033528],
        [-3.04022572, -2.96774071, -2.94935646, ..., -1.60188799,
         -1.31924672, -1.21169353],
        [-3.08627333, -2.99469027, -2.94520185, ..., -1.65119376,
         -1.37851224, -1.12608059],
        ...,
        [-5.90806618, -5.91424305, -5.83044334, ..., -6.07969807,
         -6.15040096, -6.17733007],
        [-5.96282024, -5.95189316, -5.92722682, ..., -6.08448594,
         -6.13382063, -6.08951798],
        [-5.94239881, -5.94581863, -5.89053102, ..., -6.06211703,
         -6.10507126, -6.14779285]],

       [[-6.05979383, -6.09535698, -6.09251311, ..., -2.40639559,
         -2.24981509, -2.08961257],
        [-6.08667958, -6.02456747, -6.0170815 , ..., -2.42802216,
         -2.22337436, -2.10519222],
        [-6.09931903, -6.03495442, -5.95759309, ..., -2.49291907,
         -2.2997885 , -2.12629648],
...
        [-3.32598493, -3.30382454, -3.20423784, ..., -1.00322961,
         -0.97618703, -0.95131645],
        [-3.20753121, -3.60876004, -3.49564811, ..., -1.28807735,
         -1.15578997, -1.09726923],
        [-2.93762636, -3.10258749, -3.03915487, ..., -1.47681109,
         -1.31128375, -0.95207791]],

       [[-1.11213767, -1.13699247, -1.25665864, ..., -1.35621506,
         -1.42953444, -1.4167411 ],
        [-0.98623089, -1.13964059, -1.27974119, ..., -1.0324307 ,
         -1.03130553, -1.05570056],
        [-0.99708834, -1.12944891, -1.24471766, ..., -0.65708576,
         -0.70464935, -0.73556999],
        ...,
        [-2.47702324, -2.71620081, -2.7468564 , ..., -1.40829221,
         -1.19436586, -1.01276643],
        [-3.10128252, -3.14085613, -3.33182758, ..., -1.58499368,
         -1.34399758, -1.10939919],
        [-3.51298761, -3.71845929, -3.60439832, ..., -1.69399996,
         -1.41872354, -1.10405283]]], shape=(3288, 29, 27))
Coordinates:
  * lat      (lat) float32 116B 43.25 43.0 42.75 42.5 ... 37.0 36.75 36.5 36.25
  * lon      (lon) float32 108B 267.2 267.5 267.8 268.0 ... 273.2 273.5 273.8
    county   (lat, lon) <U11 34kB dask.array<chunksize=(29, 27), meta=np.ndarray>
  * time     (time) datetime64[ns] 26kB 2016-01-01 2016-01-02 ... 2024-12-31

In [50]:
(spi_xarray < -2).sum(dim='time')

<xarray.DataArray 'SPI' (lat: 29, lon: 27)> Size: 6kB
array([[1879, 1891, 1889, 1846, 1846, 1866, 1879, 1860, 1900, 1910, 1893,
        1907, 1866, 1837, 1852, 1858, 1831, 1855, 1832, 1822, 1776, 1719,
        1716, 1668, 1624, 1576, 1598],
       [1920, 1888, 1882, 1878, 1872, 1891, 1881, 1879, 1898, 1890, 1886,
        1901, 1878, 1869, 1865, 1850, 1862, 1860, 1858, 1806, 1786, 1734,
        1709, 1681, 1625, 1562, 1557],
       [1949, 1905, 1889, 1898, 1883, 1892, 1916, 1890, 1884, 1863, 1852,
        1869, 1866, 1872, 1855, 1871, 1836, 1860, 1827, 1782, 1796, 1697,
        1685, 1669, 1613, 1576, 1554],
       [1934, 1942, 1967, 1941, 1896, 1901, 1888, 1917, 1885, 1904, 1885,
        1855, 1846, 1863, 1876, 1871, 1864, 1842, 1822, 1773, 1755, 1676,
        1650, 1635, 1594, 1547, 1486],
       [1956, 1932, 1940, 1943, 1941, 1953, 1918, 1915, 1904, 1888, 1881,
        1873, 1873, 1866, 1909, 1884, 1872, 1840, 1797, 1722, 1679, 1651,
        1647, 1627, 1552, 1521, 1526],
       [1965, 1972, 1939, 1938, 1982, 1958, 1940, 1934, 1928, 1902, 1926,
        1922, 1909, 1876, 1863, 1855, 1833, 1826, 1806, 1774, 1710, 1709,
        1660, 1590, 1544, 1512, 1495],
       [1934, 1955, 1953, 1939, 1948, 1939, 1935, 1934, 1935, 1925, 1949,
        1967, 1954, 1898, 1898, 1840, 1838, 1849, 1805, 1789, 1720, 1674,
...
        1818, 1809, 1803, 1827, 1829, 1824, 1813, 1830, 1878, 1842, 1821,
        1826, 1829, 1789, 1766, 1757],
       [1895, 1867, 1886, 1930, 1855, 1805, 1805, 1825, 1823, 1808, 1809,
        1831, 1835, 1800, 1780, 1825, 1809, 1819, 1836, 1867, 1844, 1802,
        1789, 1824, 1774, 1764, 1750],
       [1882, 1860, 1870, 1896, 1824, 1830, 1852, 1865, 1827, 1828, 1825,
        1824, 1825, 1822, 1812, 1879, 1834, 1824, 1838, 1890, 1831, 1785,
        1780, 1793, 1795, 1797, 1781],
       [1851, 1855, 1856, 1868, 1833, 1824, 1834, 1838, 1826, 1825, 1848,
        1844, 1808, 1811, 1819, 1839, 1827, 1842, 1849, 1811, 1793, 1762,
        1760, 1790, 1766, 1760, 1765],
       [1865, 1865, 1848, 1855, 1822, 1812, 1833, 1817, 1838, 1817, 1814,
        1839, 1824, 1826, 1850, 1867, 1825, 1817, 1801, 1817, 1766, 1761,
        1784, 1782, 1733, 1747, 1753],
       [1884, 1882, 1860, 1886, 1846, 1828, 1825, 1836, 1836, 1827, 1829,
        1846, 1869, 1866, 1862, 1867, 1828, 1820, 1816, 1822, 1765, 1745,
        1741, 1759, 1711, 1719, 1723],
       [1898, 1887, 1846, 1881, 1827, 1794, 1781, 1850, 1821, 1827, 1830,
        1857, 1865, 1862, 1863, 1879, 1826, 1823, 1815, 1802, 1748, 1723,
        1705, 1741, 1694, 1733, 1706]])
Coordinates:
  * lat      (lat) float32 116B 43.25 43.0 42.75 42.5 ... 37.0 36.75 36.5 36.25
  * lon      (lon) float32 108B 267.2 267.5 267.8 268.0 ... 273.2 273.5 273.8
    county   (lat, lon) <U11 34kB dask.array<chunksize=(29, 27), meta=np.ndarray>

In [41]:
tdays_extdrought


<xarray.DataArray 'TOT_DAYS_EXTDROUGHT' (lat: 29, lon: 27)> Size: 6kB
array([[2147, 2160, 2135, 2109, 2108, 2121, 2133, 2121, 2122, 2131, 2125,
        2124, 2105, 2081, 2081, 2101, 2072, 2076, 2046, 2079, 2027, 2011,
        2002, 1919, 1874, 1839, 1837],
       [2164, 2150, 2130, 2123, 2132, 2135, 2123, 2119, 2149, 2138, 2143,
        2130, 2107, 2087, 2085, 2081, 2089, 2073, 2051, 2043, 2045, 1998,
        1984, 1912, 1881, 1799, 1805],
       [2188, 2145, 2130, 2140, 2109, 2117, 2138, 2126, 2125, 2106, 2080,
        2096, 2104, 2084, 2074, 2084, 2061, 2058, 2062, 2025, 2028, 1971,
        1956, 1906, 1866, 1805, 1772],
       [2165, 2171, 2186, 2152, 2124, 2112, 2115, 2133, 2125, 2136, 2117,
        2094, 2087, 2091, 2080, 2099, 2084, 2054, 2046, 2016, 1998, 1953,
        1913, 1864, 1822, 1767, 1732],
       [2187, 2168, 2188, 2168, 2147, 2145, 2132, 2119, 2108, 2110, 2097,
        2098, 2098, 2091, 2101, 2090, 2075, 2058, 2036, 1970, 1939, 1912,
        1911, 1856, 1777, 1752, 1765],
       [2194, 2207, 2181, 2152, 2177, 2163, 2137, 2142, 2153, 2139, 2133,
        2123, 2121, 2101, 2085, 2060, 2050, 2058, 2016, 1992, 1957, 1937,
        1890, 1810, 1748, 1745, 1757],
       [2176, 2192, 2179, 2155, 2161, 2138, 2144, 2156, 2141, 2129, 2155,
        2173, 2169, 2122, 2108, 2079, 2076, 2083, 2040, 2004, 1940, 1907,
...
        2060, 2051, 2043, 2056, 2070, 2059, 2045, 2061, 2115, 2056, 2040,
        2049, 2053, 2012, 1985, 1995],
       [2128, 2084, 2097, 2140, 2089, 2048, 2022, 2052, 2060, 2034, 2045,
        2067, 2069, 2055, 2035, 2060, 2042, 2059, 2065, 2076, 2064, 2046,
        2032, 2038, 2006, 1998, 1997],
       [2108, 2081, 2091, 2120, 2046, 2070, 2075, 2088, 2070, 2084, 2091,
        2075, 2065, 2074, 2068, 2098, 2063, 2041, 2061, 2105, 2055, 2021,
        2011, 2016, 2003, 2002, 1990],
       [2084, 2085, 2092, 2090, 2058, 2054, 2065, 2059, 2064, 2058, 2085,
        2076, 2066, 2075, 2061, 2068, 2048, 2069, 2064, 2014, 2015, 1991,
        2013, 2005, 1980, 1978, 1995],
       [2107, 2108, 2075, 2091, 2062, 2038, 2054, 2052, 2062, 2067, 2053,
        2078, 2086, 2052, 2067, 2097, 2041, 2024, 2014, 2025, 1990, 1986,
        2013, 1993, 1961, 1988, 1976],
       [2129, 2110, 2115, 2113, 2092, 2070, 2073, 2071, 2070, 2061, 2060,
        2089, 2088, 2087, 2088, 2097, 2051, 2058, 2048, 2048, 2001, 1977,
        1966, 1982, 1946, 1960, 1958],
       [2121, 2138, 2094, 2108, 2079, 2047, 2037, 2081, 2061, 2059, 2059,
        2095, 2098, 2078, 2094, 2097, 2052, 2054, 2050, 2039, 1993, 1946,
        1933, 1948, 1916, 1942, 1938]])
Coordinates:
  * lat      (lat) float32 116B 43.25 43.0 42.75 42.5 ... 37.0 36.75 36.5 36.25
  * lon      (lon) float32 108B 267.2 267.5 267.8 268.0 ... 273.2 273.5 273.8
    county   (lat, lon) <U11 34kB dask.array<chunksize=(29, 27), meta=np.ndarray>

In [35]:
count_true2

<xarray.DataArray 'SPI' (lat: 29, lon: 27)> Size: 6kB
array([[2542, 2552, 2533, 2518, 2528, 2506, 2521, 2519, 2519, 2528, 2528,
        2528, 2518, 2496, 2484, 2478, 2471, 2473, 2467, 2478, 2464, 2463,
        2449, 2400, 2351, 2319, 2341],
       [2563, 2552, 2540, 2533, 2527, 2515, 2515, 2519, 2522, 2520, 2519,
        2521, 2505, 2488, 2477, 2467, 2474, 2464, 2464, 2472, 2477, 2456,
        2440, 2405, 2356, 2319, 2315],
       [2575, 2558, 2540, 2539, 2525, 2513, 2517, 2504, 2505, 2503, 2495,
        2493, 2479, 2480, 2472, 2473, 2471, 2453, 2447, 2452, 2478, 2444,
        2427, 2393, 2349, 2307, 2288],
       [2553, 2548, 2556, 2543, 2525, 2522, 2510, 2514, 2507, 2505, 2497,
        2479, 2480, 2478, 2479, 2479, 2472, 2444, 2456, 2440, 2437, 2420,
        2405, 2367, 2345, 2294, 2244],
       [2573, 2559, 2560, 2550, 2533, 2530, 2502, 2499, 2509, 2505, 2494,
        2479, 2481, 2465, 2472, 2476, 2476, 2456, 2454, 2437, 2423, 2405,
        2379, 2355, 2311, 2283, 2268],
       [2574, 2585, 2558, 2541, 2537, 2530, 2515, 2513, 2525, 2506, 2495,
        2479, 2485, 2481, 2471, 2460, 2452, 2441, 2424, 2421, 2413, 2402,
        2373, 2342, 2289, 2289, 2295],
       [2569, 2560, 2559, 2539, 2540, 2531, 2536, 2530, 2529, 2504, 2495,
        2511, 2504, 2488, 2477, 2446, 2457, 2446, 2416, 2415, 2388, 2361,
...
        2479, 2477, 2470, 2476, 2494, 2490, 2462, 2477, 2499, 2468, 2448,
        2444, 2448, 2404, 2398, 2383],
       [2493, 2481, 2483, 2486, 2468, 2448, 2438, 2449, 2450, 2451, 2461,
        2476, 2488, 2480, 2458, 2483, 2464, 2459, 2456, 2470, 2460, 2447,
        2441, 2428, 2398, 2380, 2373],
       [2474, 2477, 2481, 2474, 2465, 2480, 2485, 2491, 2486, 2478, 2500,
        2492, 2492, 2498, 2490, 2508, 2461, 2475, 2466, 2471, 2445, 2429,
        2428, 2412, 2390, 2377, 2378],
       [2486, 2491, 2493, 2487, 2480, 2480, 2486, 2479, 2481, 2472, 2491,
        2494, 2480, 2486, 2481, 2463, 2457, 2451, 2455, 2439, 2416, 2413,
        2400, 2399, 2383, 2367, 2374],
       [2499, 2511, 2496, 2490, 2484, 2475, 2481, 2490, 2472, 2472, 2464,
        2485, 2474, 2471, 2477, 2465, 2421, 2411, 2424, 2428, 2393, 2397,
        2397, 2383, 2358, 2355, 2362],
       [2522, 2508, 2503, 2500, 2496, 2487, 2494, 2493, 2480, 2469, 2478,
        2479, 2473, 2476, 2458, 2466, 2421, 2423, 2409, 2422, 2402, 2382,
        2373, 2371, 2338, 2347, 2355],
       [2518, 2524, 2518, 2504, 2493, 2480, 2474, 2479, 2488, 2490, 2458,
        2461, 2478, 2462, 2454, 2445, 2418, 2421, 2406, 2413, 2388, 2367,
        2348, 2361, 2336, 2346, 2356]])
Coordinates:
  * lat      (lat) float32 116B 43.25 43.0 42.75 42.5 ... 37.0 36.75 36.5 36.25
  * lon      (lon) float32 108B 267.2 267.5 267.8 268.0 ... 273.2 273.5 273.8
    county   (lat, lon) <U11 34kB dask.array<chunksize=(29, 27), meta=np.ndarray>

In [25]:
(spi_xarray < -0.8).groupby(['time.year', 'time.month']).sum().stack(time=('year', 'month'))#.isel(time=12)

<xarray.DataArray 'SPI' (lat: 29, lon: 27, time: 108)> Size: 677kB
array([[[28, 26, 22, ..., 29, 23, 27],
        [26, 26, 22, ..., 28, 23, 27],
        [26, 26, 22, ..., 28, 23, 27],
        ...,
        [21, 21, 20, ..., 25, 20, 16],
        [18, 21, 20, ..., 26, 20, 17],
        [19, 24, 20, ..., 26, 22, 21]],

       [[27, 26, 21, ..., 29, 23, 27],
        [28, 26, 22, ..., 28, 23, 27],
        [27, 26, 22, ..., 29, 22, 27],
        ...,
        [21, 21, 20, ..., 25, 19, 16],
        [20, 21, 20, ..., 26, 18, 15],
        [17, 23, 20, ..., 26, 21, 17]],

       [[28, 26, 21, ..., 28, 22, 28],
        [28, 26, 22, ..., 28, 22, 28],
        [28, 26, 21, ..., 28, 22, 27],
        ...,
...
        ...,
        [24, 20, 21, ..., 30, 22, 23],
        [26, 20, 20, ..., 30, 21, 23],
        [26, 19, 19, ..., 30, 21, 23]],

       [[27, 25, 22, ..., 30, 24, 23],
        [28, 25, 22, ..., 30, 24, 23],
        [28, 25, 21, ..., 30, 24, 22],
        ...,
        [24, 19, 21, ..., 30, 20, 23],
        [24, 19, 20, ..., 30, 20, 23],
        [25, 18, 19, ..., 30, 21, 23]],

       [[26, 27, 22, ..., 30, 24, 22],
        [27, 26, 21, ..., 30, 24, 22],
        [27, 26, 22, ..., 30, 24, 22],
        ...,
        [24, 20, 22, ..., 30, 20, 23],
        [24, 19, 21, ..., 30, 21, 23],
        [25, 19, 20, ..., 30, 21, 23]]], shape=(29, 27, 108))
Coordinates:
  * lat      (lat) float32 116B 43.25 43.0 42.75 42.5 ... 37.0 36.75 36.5 36.25
  * lon      (lon) float32 108B 267.2 267.5 267.8 268.0 ... 273.2 273.5 273.8
    county   (lat, lon) <U11 34kB dask.array<chunksize=(29, 27), meta=np.ndarray>
  * time     (time) object 864B MultiIndex
  * year     (time) int64 864B 2016 2016 2016 2016 2016 ... 2024 2024 2024 2024
  * month    (time) int64 864B 1 2 3 4 5 6 7 8 9 10 ... 3 4 5 6 7 8 9 10 11 12

### 3-mo SPI

In [94]:
### 3-mo SPI 
ppt = precip.where(precip > 1e-3, other=np.nan).resample(time='1ME').sum().rolling(time=3, center=True).sum() #note about rolling sum 

In [96]:
%%time 
### 3-mo SPI Gamma Fit
shape_arr = []# np.zeros((29, 27), dtype=float)
scale_arr = []# np.zeros((29, 27), dtype=float)

for la in np.arange(0,29):
    for lo in np.arange(0,27):
        data_flat = ds.isel(lat=la,lon=lo).values.flatten()
        data_flat = data_flat[~np.isnan(data_flat)]

        # Fit gamma distribution fixing location to 0
        %time shape, _, scale = stats.gamma.fit(data_flat, floc=0)
        shape_arr.append(shape)
        scale_arr.append(scale)

CPU times: user 232 μs, sys: 0 ns, total: 232 μs
Wall time: 243 μs
CPU times: user 137 μs, sys: 50 μs, total: 187 μs
Wall time: 192 μs
CPU times: user 194 μs, sys: 0 ns, total: 194 μs
Wall time: 206 μs
CPU times: user 124 μs, sys: 45 μs, total: 169 μs
Wall time: 174 μs
CPU times: user 195 μs, sys: 0 ns, total: 195 μs
Wall time: 207 μs
CPU times: user 126 μs, sys: 45 μs, total: 171 μs
Wall time: 177 μs
CPU times: user 176 μs, sys: 0 ns, total: 176 μs
Wall time: 187 μs
CPU times: user 126 μs, sys: 45 μs, total: 171 μs
Wall time: 177 μs
CPU times: user 126 μs, sys: 46 μs, total: 172 μs
Wall time: 177 μs
CPU times: user 172 μs, sys: 0 ns, total: 172 μs
Wall time: 177 μs
CPU times: user 192 μs, sys: 0 ns, total: 192 μs
Wall time: 203 μs
CPU times: user 127 μs, sys: 46 μs, total: 173 μs
Wall time: 178 μs
CPU times: user 126 μs, sys: 46 μs, total: 172 μs
Wall time: 177 μs
CPU times: user 149 μs, sys: 54 μs, total: 203 μs
Wall time: 209 μs
CPU times: user 127 μs, sys: 46 μs, total: 173 μs
Wall

In [99]:
gamma_shape = np.tile(np.array(shape_arr).reshape(29,27), (len(ds.time), 1, 1))
gamma_scale = np.tile(np.array(scale_arr).reshape(29,27), (len(ds.time), 1, 1))

In [100]:
# Step 2: Calculate CDF for each day in original data
probabilities = stats.gamma.cdf(ds, a=gamma_shape, loc=0, scale=gamma_scale)
# Step 3: Transform probabilities to a standard normal distribution (Z-score)
spi_values = stats.norm.ppf(probabilities)

In [144]:
data_flat = ds.values.flatten()
data_flat = data_flat[~np.isnan(data_flat)]

# Fit gamma distribution fixing location to 0
shape, loc, scale = stats.gamma.fit(data_flat, floc=0)


# Step 2: Calculate CDF for each day in original data
probabilities = stats.gamma.cdf(ds, a=shape, loc=loc, scale=scale)
# Step 3: Transform probabilities to a standard normal distribution (Z-score)
spi_values = stats.norm.ppf(probabilities)

# spi_values is an ndarray, convert back to xarray DataArray with original coords and dims
spi_xarray = xr.DataArray(
    spi_values,
    coords=ds.coords,
    dims=ds.dims,
    name='SPI'
)

In [142]:
ds

<xarray.DataArray 'precip' (time: 3288, lat: 29, lon: 27)> Size: 10MB
dask.array<where, shape=(3288, 29, 27), dtype=float32, chunksize=(3288, 29, 27), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float32 116B 43.25 43.0 42.75 42.5 ... 37.0 36.75 36.5 36.25
  * lon      (lon) float32 108B 267.2 267.5 267.8 268.0 ... 273.2 273.5 273.8
    county   (lat, lon) <U11 34kB dask.array<chunksize=(29, 27), meta=np.ndarray>
  * time     (time) datetime64[ns] 26kB 2016-01-01 2016-01-02 ... 2024-12-31
Attributes:
    long_name:   Total precipitation
    short_name:  tp
    units:       m

In [ ]:
def fit_gamma_1d(data_1d):
    # Remove non-finite values to avoid fitting errors
    finite_data = data_1d[np.isfinite(data_1d)]
    if len(finite_data) == 0:
        return np.array([np.nan, np.nan, np.nan])  # Return NaNs if no data
    shape, loc, scale = stats.gamma.fit(finite_data, floc=0)
    return np.array([shape, loc, scale])

# Assume precip_data is xarray DataArray with dims ('time', 'lat', 'lon')
params = xr.apply_ufunc(
    fit_gamma_1d,
    precip_data,
    input_core_dims=[['time']],
    output_core_dims=[['param']],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float],
    output_sizes={'param': 3}
)

# Rename 'param' dimension to meaningful parameter names
params['param'] = ['shape', 'loc', 'scale']

In [140]:
np.nanmin(spi_xarray)

np.float64(-1.3987959510729442)

In [141]:
np.nanmax(spi_xarray)

np.float64(5.982927502233968)

In [130]:
tdays_moddrought = (spi_xarray < -0.8).groupby(['time.year', 'time.month']).sum().stack(time=('year', 'month')).rename('TOT_DAYS_MODDROUGHT')
tdays_extdrought = (spi_xarray < -1.6).groupby(['time.year', 'time.month']).sum().stack(time=('year', 'month')).rename('TOT_DAYS_EXTDROUGHT')
tdays_modplv = (spi_xarray > 0.8).groupby(['time.year', 'time.month']).sum().stack(time=('year', 'month')).rename('TOT_DAYS_MODPLV')
tdays_extplv = (spi_xarray > 1.6).groupby(['time.year', 'time.month']).sum().stack(time=('year', 'month')).rename('TOT_DAYS_EXTPLV')

# avgdays_moddrought = (spi < -0.8).groupby(['time.year', 'time.month']).mean().stack(time=('year', 'month')).rename('AVG_DAYS_MODDROUGHT')
# avgdays_extdrought = (spi < -1.6).groupby(['time.year', 'time.month']).mean().stack(time=('year', 'month')).rename('AVG_DAYS_EXTDROUGHT')
# avgdays_modplv = (spi > 0.8).groupby(['time.year', 'time.month']).mean().stack(time=('year', 'month')).rename('AVG_DAYS_MODPLV')
# avgdays_extplv = (spi > 1.6).groupby(['time.year', 'time.month']).mean().stack(time=('year', 'month')).rename('AVG_DAYS_EXTPLV')

monthlystat = xr.merge([tdays_moddrought,tdays_extdrought,tdays_modplv,tdays_extplv])
times = pd.to_datetime(['{}-{:02d}-01'.format(y, m) for y, m in monthlystat['time'].values])
monthlystat = monthlystat.drop_vars(['time', 'year', 'month']).assign_coords(time=times)

# monthlystat.to_netcdf('/data/keeling/a/rytam2/iema/data/output/monthlystats_spi_latlon_2016-2025.csv')

In [137]:
np.unique(monthlystat['TOT_DAYS_EXTPLV'])

array([0, 1, 2, 3, 4, 5, 6, 7, 8])

In [63]:
spi.isel(time=152,lat=23).values

array([-0.55345094, -0.38867766, -0.26684216, -0.37762377, -0.11096112,
        0.1300419 ,  0.24452977, -0.00582162, -0.27769655, -0.36343405,
       -0.44637308, -0.48022386, -0.6981922 , -0.67036635, -0.56327486,
       -0.59243035, -0.7312801 , -0.80544645, -0.86674654, -0.9914542 ,
       -0.84277487, -0.7983417 , -0.78431296, -0.74643475, -1.1080511 ,
       -0.9946813 , -1.0001106 ], dtype=float32)

In [71]:
np.max(spi.values)

np.float32(8.446438)

In [72]:
np.min(spi.values)

np.float32(-1.3542869)

In [136]:
monthlystat['TOT_DAYS_EXTPLV'].isel(time=15,lat=23)

<xarray.DataArray 'TOT_DAYS_EXTPLV' (lon: 27)> Size: 216B
array([7, 7, 7, 7, 7, 7, 7, 6, 6, 6, 7, 6, 6, 5, 3, 2, 2, 2, 1, 1, 1, 1,
       2, 2, 2, 1, 1])
Coordinates:
    lat      float32 4B 37.5
  * lon      (lon) float32 108B 267.2 267.5 267.8 268.0 ... 273.2 273.5 273.8
    county   (lon) <U11 1kB dask.array<chunksize=(27,), meta=np.ndarray>
    time     datetime64[ns] 8B 2017-04-01

In [74]:
monthlystat['TOT_DAYS_EXTDROUGHT'].isel(time=12,lon=12,).values

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0])

In [126]:
(spi > 0.8).groupby(['time.year']).sum()

<xarray.DataArray 'precip' (year: 10, lat: 29, lon: 27)> Size: 63kB
dask.array<transpose, shape=(10, 29, 27), dtype=int64, chunksize=(1, 29, 27), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float32 116B 43.25 43.0 42.75 42.5 ... 37.0 36.75 36.5 36.25
  * lon      (lon) float32 108B 267.2 267.5 267.8 268.0 ... 273.2 273.5 273.8
    county   (lat, lon) <U11 34kB dask.array<chunksize=(29, 27), meta=np.ndarray>
  * year     (year) int64 80B 2016 2017 2018 2019 2020 2021 2022 2023 2024 2025

In [118]:
moddrought = (spi < -0.8).groupby(['time.year', 'time.month']).sum().stack(time=('year', 'month')).rename('TOT_DAYS_MODDROUGHT')
extdrought = (spi < -1.6).groupby(['time.year', 'time.month']).sum().stack(time=('year', 'month')).rename('TOT_DAYS_EXTDROUGHT')
modplv = (spi > 0.8).groupby(['time.year', 'time.month']).sum().stack(time=('year', 'month')).rename('TOT_DAYS_MODPLV')
extplv = (spi > 1.6).groupby(['time.year', 'time.month']).sum().stack(time=('year', 'month')).rename('TOT_DAYS_EXTPLV')


<xarray.DataArray 'precip' (lat: 29, lon: 27, time: 120)> Size: 752kB
dask.array<reshape, shape=(29, 27, 120), dtype=int64, chunksize=(29, 27, 1), chunktype=numpy.ndarray>
Coordinates:
  * lat      (lat) float32 116B 43.25 43.0 42.75 42.5 ... 37.0 36.75 36.5 36.25
  * lon      (lon) float32 108B 267.2 267.5 267.8 268.0 ... 273.2 273.5 273.8
    county   (lat, lon) <U11 34kB dask.array<chunksize=(29, 27), meta=np.ndarray>
  * time     (time) datetime64[ns] 960B 2016-01-01 2016-02-01 ... 2025-12-01

In [84]:
a = monthlystat.values.ravel()

array([2, 1, 7, ..., 1, 0, 0], shape=(93960,))